# PHASE 1

In [ ]:
import torch #importing the pytorch to do transformer model
print(torch.cuda.is_available()) # checks weather CUDA is available CUDA = NVIDIA GPU computing platform


True


In [ ]:
!nvidia-smi #nvidia system management interface example show me detailed GPU stats

Sat May 16 06:15:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install transformer_lens
!pip install circuitsvis
!pip install plotly einops jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.8 MB/s eta 0:00:00


In [ ]:
import transformer_lens
import circuitsvis
import plotly
import einops
import jaxtyping

print("Setup successful!")

Setup successful!


In [ ]:
import transformer_lens as tl

model = tl.HookedTransformer.from_pretrained("gpt2")
model.cfg

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer


HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 84


```
Input Tokens
     ↓
Embedding Layer
     ↓
[Layer 1]
  Attention
  MLP
     ↓
[Layer 2]
  Attention
  MLP
     ↓
...
     ↓
[Layer 12]
     ↓
Output Probabilities
```


In [ ]:
tokens = model.to_tokens("The Eiffel Tower is in")
logits, cache = model.run_with_cache(tokens)

# See the top prediction
next_token = logits[0, -1].argmax()
print(model.to_string(next_token))

 the


In [ ]:
import torch

for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]

    logits_at_layer = model.unembed(
        model.ln_final(resid.unsqueeze(0).unsqueeze(0))
    )

    top_token = logits_at_layer[0, 0].argmax()

    print(f"Layer {layer:2d}: {model.to_string(top_token)}")

Layer  0:  the
Layer  1:  order
Layer  2:  order
Layer  3:  order
Layer  4:  front
Layer  5:  the
Layer  6:  the
Layer  7:  front
Layer  8:  a
Layer  9:  London
Layer 10:  the
Layer 11:  the


In [ ]:
import circuitsvis as cv

# Attention pattern for layer 0
attn = cache["pattern", 0]  # shape: [batch, heads, seq, seq]

cv.attention.attention_patterns(
    tokens=model.to_str_tokens("The Eiffel Tower is in"),
    attention=attn[0]
)

In [ ]:
tokens = model.to_tokens("Alice gave Bob the book because")
logits, cache = model.run_with_cache(tokens)

# See the top prediction
next_token = logits[0, -1].argmax()
print(model.to_string(next_token))

 she


In [ ]:
import torch

for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]

    logits_at_layer = model.unembed(
        model.ln_final(resid.unsqueeze(0).unsqueeze(0))
    )

    top_token = logits_at_layer[0, 0].argmax()

    print(f"Layer {layer:2d}: {model.to_string(top_token)}")

Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  they
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she


In [ ]:
import circuitsvis as cv

# Attention pattern for layer 0
attn = cache["pattern", 0]  # shape: [batch, heads, seq, seq]

cv.attention.attention_patterns(
    tokens=model.to_str_tokens("Alice gave Bob the book because"),
    attention=attn[0]
)

In [ ]:
tokens = model.to_tokens("The capital of France is")
logits, cache = model.run_with_cache(tokens)

# See the top prediction
next_token = logits[0, -1].argmax()
print(model.to_string(next_token))

 now


In [ ]:
import torch

for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]

    logits_at_layer = model.unembed(
        model.ln_final(resid.unsqueeze(0).unsqueeze(0))
    )

    top_token = logits_at_layer[0, 0].argmax()

    print(f"Layer {layer:2d}: {model.to_string(top_token)}")

In [ ]:
import circuitsvis as cv

# Attention pattern for layer 0
attn = cache["pattern", 0]  # shape: [batch, heads, seq, seq]

cv.attention.attention_patterns(
    tokens=model.to_str_tokens("The capital of France is"),
    attention=attn[0]
)

In [ ]:
import torch

for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]

    logits_at_layer = model.unembed(
        model.ln_final(resid.unsqueeze(0).unsqueeze(0))
    )

    top_token = logits_at_layer[0, 0].argmax()

    print(f"Layer {layer:2d}: {model.to_string(top_token)}")

Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  they
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she


In [ ]:
tokens = model.to_tokens("2 + 2 =")
logits, cache = model.run_with_cache(tokens)

# See the top prediction
next_token = logits[0, -1].argmax()
print(model.to_string(next_token))

 2


In [ ]:
import circuitsvis as cv

# Attention pattern for layer 0
attn = cache["pattern", 0]  # shape: [batch, heads, seq, seq]

cv.attention.attention_patterns(
    tokens=model.to_str_tokens("2 + 2 ="),
    attention=attn[0]
)

In [ ]:
import torch

for layer in range(model.cfg.n_layers):
    resid = cache["resid_post", layer][0, -1]

    logits_at_layer = model.unembed(
        model.ln_final(resid.unsqueeze(0).unsqueeze(0))
    )

    top_token = logits_at_layer[0, 0].argmax()

    print(f"Layer {layer:2d}: {model.to_string(top_token)}")

Layer  0: ========
Layer  1: ========
Layer  2: ========
Layer  3: ========
Layer  4: ========
Layer  5: ========
Layer  6: ========
Layer  7: ========
Layer  8:  0
Layer  9:  2
Layer 10:  2
Layer 11:  2


In [ ]:
import transformer_lens as tl
import circuitsvis as cv
import torch

# Load model
model = tl.HookedTransformer.from_pretrained("gpt2")

# Prompts to investigate
prompts = [
    "The capital of France is",
    "2 + 2 =",
    "The Eiffel Tower is in",
    "Alice gave Bob the book because",
    "Once upon a",
    "The dogs in the yard are",
]

for prompt in prompts:

    print("\n" + "=" * 60)
    print(f"PROMPT: {prompt}")
    print("=" * 60)

    # Tokenize
    tokens = model.to_tokens(prompt)

    # Run model and cache activations
    logits, cache = model.run_with_cache(tokens)

    # Final prediction
    next_token = logits[0, -1].argmax()
    prediction = model.to_string(next_token)

    print(f"\nFinal Prediction: {prediction}")

    # Logit Lens
    print("\nLogit Lens:")

    for layer in range(model.cfg.n_layers):

        resid = cache["resid_post", layer][0, -1]

        logits_at_layer = model.unembed(
            model.ln_final(
                resid.unsqueeze(0).unsqueeze(0)
            )
        )

        top_token = logits_at_layer[0, 0].argmax()

        print(
            f"Layer {layer:2d}: "
            f"{model.to_string(top_token)}"
        )

    # Attention visualization for Layer 0
    print("\nShowing attention patterns for Layer 0...")

    attn = cache["pattern", 0]

    display(
        cv.attention.attention_patterns(
            tokens=model.to_str_tokens(prompt),
            attention=attn[0]
        )
    )

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

PROMPT: The capital of France is

Final Prediction:  now

Logit Lens:
Layer  0:  not
Layer  1:  now
Layer  2:  now
Layer  3:  now
Layer  4:  now
Layer  5:  now
Layer  6:  now
Layer  7:  now
Layer  8:  now
Layer  9:  now
Layer 10:  now
Layer 11:  now

Showing attention patterns for Layer 0...



PROMPT: 2 + 2 =

Final Prediction:  2

Logit Lens:
Layer  0: ========
Layer  1: ========
Layer  2: ========
Layer  3: ========
Layer  4: ========
Layer  5: ========
Layer  6: ========
Layer  7: ========
Layer  8:  0
Layer  9:  2
Layer 10:  2
Layer 11:  2

Showing attention patterns for Layer 0...



PROMPT: The Eiffel Tower is in

Final Prediction:  the

Logit Lens:
Layer  0:  the
Layer  1:  order
Layer  2:  order
Layer  3:  order
Layer  4:  front
Layer  5:  the
Layer  6:  the
Layer  7:  front
Layer  8:  a
Layer  9:  London
Layer 10:  the
Layer 11:  the

Showing attention patterns for Layer 0...



PROMPT: Alice gave Bob the book because

Final Prediction:  she

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  they
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she

Showing attention patterns for Layer 0...



PROMPT: Once upon a

Final Prediction:  time

Logit Lens:
Layer  0:  few
Layer  1:  few
Layer  2:  few
Layer  3:  couple
Layer  4:  few
Layer  5:  moment
Layer  6:  few
Layer  7:  few
Layer  8:  time
Layer  9:  time
Layer 10:  time
Layer 11:  time

Showing attention patterns for Layer 0...



PROMPT: The dogs in the yard are

Final Prediction:  not

Logit Lens:
Layer  0:  not
Layer  1:  not
Layer  2:  not
Layer  3:  not
Layer  4:  not
Layer  5:  not
Layer  6:  not
Layer  7:  not
Layer  8:  not
Layer  9:  adorable
Layer 10:  dogs
Layer 11:  not

Showing attention patterns for Layer 0...


In [ ]:
import transformer_lens as tl
import circuitsvis as cv
import torch

# Load model
model = tl.HookedTransformer.from_pretrained("gpt2")

# Prompts to investigate
prompts = [
    "Alice thanked Bob because",
    "Alice yelled at Bob because",
    "John helped Mary because",
    "Sarah called Emma because",
    "When Alice and Bob went to the park, Alice handed the book to Bob because",
]

for prompt in prompts:

    print("\n" + "=" * 60)
    print(f"PROMPT: {prompt}")
    print("=" * 60)

    # Tokenize
    tokens = model.to_tokens(prompt)

    # Run model and cache activations
    logits, cache = model.run_with_cache(tokens)

    # Final prediction
    next_token = logits[0, -1].argmax()
    prediction = model.to_string(next_token)

    print(f"\nFinal Prediction: {prediction}")

    # Logit Lens
    print("\nLogit Lens:")

    for layer in range(model.cfg.n_layers):

        resid = cache["resid_post", layer][0, -1]

        logits_at_layer = model.unembed(
            model.ln_final(
                resid.unsqueeze(0).unsqueeze(0)
            )
        )

        top_token = logits_at_layer[0, 0].argmax()

        print(
            f"Layer {layer:2d}: "
            f"{model.to_string(top_token)}"
        )

    # Attention visualization for Layer 0
    print("\nShowing attention patterns for Layer 0...")

    attn = cache["pattern", 0]

    display(
        cv.attention.attention_patterns(
            tokens=model.to_str_tokens(prompt),
            attention=attn[0]
        )
    )

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer

PROMPT: Alice thanked Bob because

Final Prediction:  he

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  the
Layer  4:  he
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  he
Layer 11:  he

Showing attention patterns for Layer 0...



PROMPT: Alice yelled at Bob because

Final Prediction:  he

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  he
Layer  5:  he
Layer  6:  his
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  he
Layer 11:  he

Showing attention patterns for Layer 0...



PROMPT: John helped Mary because

Final Prediction:  he

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  the
Layer  4:  the
Layer  5:  he
Layer  6:  she
Layer  7:  he
Layer  8:  she
Layer  9:  she
Layer 10:  she
Layer 11:  he

Showing attention patterns for Layer 0...



PROMPT: Sarah called Emma because

Final Prediction:  she

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  the
Layer  4:  they
Layer  5:  he
Layer  6:  she
Layer  7:  she
Layer  8:  she
Layer  9:  she
Layer 10:  she
Layer 11:  she

Showing attention patterns for Layer 0...



PROMPT: When Alice and Bob went to the park, Alice handed the book to Bob because

Final Prediction:  she

Logit Lens:
Layer  0:  it
Layer  1:  the
Layer  2:  they
Layer  3:  they
Layer  4:  they
Layer  5:  they
Layer  6:  they
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she

Showing attention patterns for Layer 0...


# PHASE 2

In [ ]:
prompts = [

    # Basic transfer / helping
    "Alice gave Bob the book because ___ was kind",          # she
    "Bob helped Alice move because ___ was strong",          # he
    "Sarah thanked John because ___ was helpful",            # he
    "John comforted Sarah because ___ was upset",            # she
    "Emma supported James because ___ was struggling",       # he
    "James encouraged Emma because ___ was nervous",         # she

    # Communication
    "When Alice told Bob the news, ___ was shocked",         # he
    "When Bob explained the mistake to Alice, ___ understood immediately",  # she
    "Sarah informed John about the meeting because ___ forgot",             # he
    "James reminded Emma about the deadline because ___ was late",          # she

    # Emotional / social
    "Alice hugged Bob because ___ was sad",                  # he
    "Bob admired Alice because ___ was talented",            # she
    "Sarah respected John because ___ worked hard",          # he
    "John trusted Sarah because ___ was honest",             # she
    "Emma apologized to James because ___ made a mistake",   # she
    "James forgave Emma because ___ felt guilty",            # she

    # Object transfer
    "Alice handed the keys to Bob because ___ was leaving",  # he
    "Bob returned the laptop to Alice because ___ needed it", # she
    "Sarah passed the documents to John because ___ requested them", # he
    "James mailed the package to Emma because ___ was waiting for it", # she

    # Long-context examples
    "When Alice and Bob arrived at the station, Alice gave the tickets to Bob because ___ had the bags", # he
    "After Sarah met John at the office, Sarah brought coffee for John because ___ looked tired", # he
    "When Emma and James finished the project, Emma thanked James because ___ solved the hardest problem", # he
    "After Bob visited Alice at home, Bob cooked dinner for Alice because ___ was exhausted", # she

    # Reversed order / harder cases
    "Bob received advice from Alice because ___ had more experience", # she
    "Alice learned programming from Bob because ___ was an expert",   # he
    "James borrowed notes from Emma because ___ missed class",        # he
    "Emma trusted James because ___ always kept promises",            # he

    # Ambiguous / reasoning-heavy
    "Alice criticized Bob because ___ made a careless mistake",       # he
    "Bob praised Alice because ___ solved the problem quickly",       # she
    "Sarah warned John because ___ looked distracted",                # he
    "John thanked Sarah because ___ stayed late to help",             # she

]

In [ ]:
dataset = [

    # Basic transfer / helping
    {
        "prompt": "Alice gave Bob the book because",
        "correct": " she",
        "incorrect": " he"
    },
    {
        "prompt": "Bob helped Alice move because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "Sarah thanked John because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "John comforted Sarah because",
        "correct": " she",
        "incorrect": " he"
    },

    # Communication
    {
        "prompt": "When Alice told Bob the news,",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "When Bob explained the mistake to Alice,",
        "correct": " she",
        "incorrect": " he"
    },
    {
        "prompt": "Sarah informed John about the meeting because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "James reminded Emma about the deadline because",
        "correct": " she",
        "incorrect": " he"
    },

    # Emotional / social
    {
        "prompt": "Alice hugged Bob because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "Bob admired Alice because",
        "correct": " she",
        "incorrect": " he"
    },

    # Long-context examples
    {
        "prompt": "When Alice and Bob arrived at the station, Alice gave the tickets to Bob because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "After Sarah met John at the office, Sarah brought coffee for John because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "When Emma and James finished the project, Emma thanked James because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "After Bob visited Alice at home, Bob cooked dinner for Alice because",
        "correct": " she",
        "incorrect": " he"
    },

    # Reversed order / harder cases
    {
        "prompt": "Bob received advice from Alice because",
        "correct": " she",
        "incorrect": " he"
    },
    {
        "prompt": "Alice learned programming from Bob because",
        "correct": " he",
        "incorrect": " she"
    },

    # Ambiguous / reasoning-heavy
    {
        "prompt": "Alice criticized Bob because",
        "correct": " he",
        "incorrect": " she"
    },
    {
        "prompt": "Bob praised Alice because",
        "correct": " she",
        "incorrect": " he"
    },

]

In [ ]:
from transformer_lens import patching
import plotly.express as px
import torch

In [ ]:
example = dataset[0]

clean_prompt = example["prompt"]
correct_answer = example["correct"]
incorrect_answer = example["incorrect"]

In [ ]:
corrupted_prompt = clean_prompt.replace(
    "Alice", "TEMP"
).replace(
    "Bob", "Alice"
).replace(
    "TEMP", "Bob"
)

In [ ]:
print("Clean Prompt:")
print(clean_prompt)

print("\nCorrupted Prompt:")
print(corrupted_prompt)

Clean Prompt:
Alice gave Bob the book because

Corrupted Prompt:
Bob gave Alice the book because


In [ ]:
clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)

In [ ]:
clean_logits, clean_cache = model.run_with_cache(clean_tokens)

In [ ]:
correct_token = model.to_single_token(correct_answer)
incorrect_token = model.to_single_token(incorrect_answer)

def metric(logits):

    final_logits = logits[0, -1]

    return (
        final_logits[correct_token]
        - final_logits[incorrect_token]
    )

In [ ]:
results = patching.get_act_patch_attn_head_out_all_pos(
    model,
    corrupted_tokens,
    clean_cache,
    metric
)

  0%|          | 0/144 [00:00<?, ?it/s]

In [ ]:
fig = px.imshow(
    results.detach().cpu().numpy(),

    labels={
        "x": "Head",
        "y": "Layer",
        "color": "Patch Effect"
    },

    title="Causal Importance of Attention Heads"
)

fig.show()

In [ ]:
# Choose the important head from the heatmap
layer = 10
head = 9

# Visualize attention pattern for that specific head
cv.attention.attention_patterns(
    tokens=model.to_str_tokens(clean_prompt),

    attention=clean_cache["pattern", layer][0, head:head+1]
)

In [ ]:
# Choose the head identified from activation patching
layer = 10
head_idx = 9

# Use one of your pronoun-resolution prompts
prompt = "When Alice and Bob went to the park, Alice handed the book to Bob because"

tokens = model.to_tokens(prompt)

# Baseline prediction
baseline_logits = model(tokens)

baseline_prediction = model.to_string(
    baseline_logits[0, -1].argmax()
)

print("Baseline Prediction:")
print(baseline_prediction)

# ----------------------------------------
# Head Ablation
# ----------------------------------------

def ablate_head(value, hook):

    # value shape:
    # [batch, seq, head, d_head]

    value[:, :, head_idx, :] = 0

    return value

# Run model with the head ablated
ablated_logits = model.run_with_hooks(

    tokens,

    fwd_hooks=[
        (
            f"blocks.{layer}.attn.hook_v",
            ablate_head
        )
    ]
)

# Prediction after ablation
ablated_prediction = model.to_string(
    ablated_logits[0, -1].argmax()
)

print("\nPrediction After Ablation:")
print(ablated_prediction)

# ----------------------------------------
# Compare Pronoun Logits
# ----------------------------------------

she_token = model.to_single_token(" she")
he_token = model.to_single_token(" he")

baseline_she = baseline_logits[0, -1, she_token].item()
baseline_he = baseline_logits[0, -1, he_token].item()

ablated_she = ablated_logits[0, -1, she_token].item()
ablated_he = ablated_logits[0, -1, he_token].item()

print("\nBaseline:")
print("she logit:", baseline_she)
print("he   logit:", baseline_he)

print("\nAfter Ablation:")
print("she logit:", ablated_she)
print("he   logit:", ablated_he)

print("\nPreference Difference:")
print("Baseline:", baseline_she - baseline_he)
print("Ablated :", ablated_she - ablated_he)

Baseline Prediction:
 she

Prediction After Ablation:
 he

Baseline:
she logit: 18.086389541625977
he   logit: 17.95142936706543

After Ablation:
she logit: 17.36487579345703
he   logit: 18.015642166137695

Preference Difference:
Baseline: 0.13496017456054688
Ablated : -0.6507663726806641


In [ ]:
from transformer_lens import patching
import torch
import plotly.express as px

# Store cumulative results
all_results = []

for example in dataset:

    clean_prompt = example["prompt"]

    correct_answer = example["correct"]
    incorrect_answer = example["incorrect"]

    # ----------------------------------------
    # Create corrupted prompt
    # ----------------------------------------

    corrupted_prompt = clean_prompt.replace(
        "Alice", "TEMP"
    ).replace(
        "Bob", "Alice"
    ).replace(
        "TEMP", "Bob"
    )

    # ----------------------------------------
    # Tokenize
    # ----------------------------------------

    clean_tokens = model.to_tokens(clean_prompt)
    corrupted_tokens = model.to_tokens(corrupted_prompt)

    # ----------------------------------------
    # Run clean prompt
    # ----------------------------------------

    clean_logits, clean_cache = model.run_with_cache(
        clean_tokens
    )

    # ----------------------------------------
    # Metric function
    # ----------------------------------------

    correct_token = model.to_single_token(
        correct_answer
    )

    incorrect_token = model.to_single_token(
        incorrect_answer
    )

    def metric(logits):

        final_logits = logits[0, -1]

        return (
            final_logits[correct_token]
            - final_logits[incorrect_token]
        )

    # ----------------------------------------
    # Activation patching
    # ----------------------------------------

    results = patching.get_act_patch_attn_head_out_all_pos(

        model,
        corrupted_tokens,
        clean_cache,
        metric

    )

    all_results.append(results)

# ----------------------------------------
# Average across dataset
# ----------------------------------------

mean_results = torch.stack(all_results).mean(dim=0)

# ----------------------------------------
# Visualize heatmap
# ----------------------------------------

fig = px.imshow(

    mean_results.detach().cpu().numpy(),

    labels={
        "x": "Head",
        "y": "Layer",
        "color": "Average Patch Effect"
    },

    title="Average Attention Head Importance Across Dataset"

)

fig.show()

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

In [ ]:
import transformer_lens.patching as patching
import torch

def logit_diff_metric(logits, correct_token, incorrect_token):

    correct_id = model.to_single_token(correct_token)
    incorrect_id = model.to_single_token(incorrect_token)

    return (
        logits[0, -1, correct_id]
        - logits[0, -1, incorrect_id]
    )

results_per_prompt = []

for item in dataset:

    clean_tokens = model.to_tokens(item["prompt"])

    # Corrupt prompt by swapping Alice/Bob
    corrupted_prompt = item["prompt"].replace(
        "Alice", "TEMP"
    ).replace(
        "Bob", "Alice"
    ).replace(
        "TEMP", "Bob"
    )

    corrupted_tokens = model.to_tokens(corrupted_prompt)

    # Run clean prompt
    clean_logits, clean_cache = model.run_with_cache(
        clean_tokens
    )

    # Activation patching
    patch_results = patching.get_act_patch_attn_head_out_all_pos(

        model,
        corrupted_tokens,
        clean_cache,

        lambda logits: logit_diff_metric(
            logits,
            item["correct"],
            item["incorrect"]
        )

    )

    results_per_prompt.append(patch_results)

# Average across dataset
avg_patch_results = torch.stack(
    results_per_prompt
).mean(0)

print("Shape:", avg_patch_results.shape)

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

Shape: torch.Size([12, 12])


In [ ]:
import plotly.express as px
import pandas as pd

fig = px.imshow(
    avg_patch_results.cpu().numpy(),
    labels=dict(x="Head", y="Layer", color="Logit diff restored"),
    title="Which heads are causally responsible for pronoun resolution?",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0
)
fig.show()

"Causal pronoun resolution in GPT-2 is mediated primarily by two heads in layer 10 (heads 7 and 10), with an earlier signal at layer 6 head 0. The intervening layers contribute minimally, suggesting a sparse two-stage circuit rather than distributed processing."

In [ ]:
# Print top 5 most negative (most causally important) heads
flat = avg_patch_results.flatten()
top_indices = flat.argsort()[:5]
for idx in top_indices:
    layer = idx // model.cfg.n_heads
    head = idx % model.cfg.n_heads
    print(f"Layer {layer}, Head {head}: {flat[idx]:.4f}")

Layer 11, Head 8: -0.2671
Layer 10, Head 9: -0.2386
Layer 6, Head 0: -0.1722
Layer 10, Head 7: -0.1555
Layer 11, Head 1: -0.1391


In [ ]:
def ablate_head_hook(layer, head):
    def hook(value, hook):
        value[:, :, head, :] = 0
        return value
    return hook

# Test: does removing L10H10 break pronoun resolution?
logits_ablated = model.run_with_hooks(
    model.to_tokens("Alice gave Bob the book because"),
    fwd_hooks=[(f"blocks.10.attn.hook_v", ablate_head_hook(10, 10))]
)
print(model.to_string(logits_ablated[0, -1].argmax()))

 she


In [ ]:
# Ablate all top heads simultaneously
top_heads = [(11, 8), (10, 9), (6, 0), (10, 7), (11, 1)]

hooks = []
for layer, head in top_heads:
    hooks.append((f"blocks.{layer}.attn.hook_v", ablate_head_hook(layer, head)))

logits_ablated = model.run_with_hooks(
    model.to_tokens("Alice gave Bob the book because"),
    fwd_hooks=hooks
)
print("All top heads ablated:", model.to_string(logits_ablated[0, -1].argmax()))

# Also test on multiple prompts
for item in dataset[:6]:
    tokens = model.to_tokens(item["prompt"])
    logits_ablated = model.run_with_hooks(tokens, fwd_hooks=hooks)
    pred = model.to_string(logits_ablated[0, -1].argmax())
    correct = pred.strip() == item["correct"].strip()
    print(f"{'✓' if correct else '✗'} '{item['prompt']}' → '{pred}' (expected '{item['correct']}')")

All top heads ablated:  he
✗ 'Alice gave Bob the book because' → ' he' (expected ' she')
✗ 'Bob helped Alice move because' → ' Alice' (expected ' he')
✓ 'Sarah thanked John because' → ' he' (expected ' he')
✓ 'John comforted Sarah because' → ' she' (expected ' she')
✗ 'When Alice told Bob the news,' → ' Alice' (expected ' he')
✗ 'When Bob explained the mistake to Alice,' → ' Alice' (expected ' she')


In [ ]:
correct_count = 0
total = len(dataset)

# Baseline accuracy (no ablation)
for item in dataset:
    tokens = model.to_tokens(item["prompt"])
    logits, _ = model.run_with_cache(tokens)
    pred = model.to_string(logits[0, -1].argmax())
    if pred.strip() == item["correct"].strip():
        correct_count += 1

baseline_acc = correct_count / total
print(f"Baseline accuracy: {baseline_acc:.1%}")

# Ablated accuracy
correct_ablated = 0
for item in dataset:
    tokens = model.to_tokens(item["prompt"])
    logits_ablated = model.run_with_hooks(tokens, fwd_hooks=hooks)
    pred = model.to_string(logits_ablated[0, -1].argmax())
    if pred.strip() == item["correct"].strip():
        correct_ablated += 1

ablated_acc = correct_ablated / total
print(f"Ablated accuracy:  {ablated_acc:.1%}")
print(f"Drop:              {baseline_acc - ablated_acc:.1%}")

Baseline accuracy: 77.8%
Ablated accuracy:  72.2%
Drop:              5.6%


In [ ]:
# ============================================================
# Mechanistic Interpretability — Pronoun Resolution Circuit
# GPT-2 Small + TransformerLens
# SAFE VERSION (handles token-length mismatches)
# ============================================================


# ============================================================
# CELL 1 — Install Libraries
# ============================================================

!pip install transformer_lens
!pip install circuitsvis
!pip install plotly einops jaxtyping


# ============================================================
# CELL 2 — Imports
# ============================================================

import torch
import transformer_lens as tl
import transformer_lens.patching as patching
import circuitsvis as cv
import plotly.express as px

from IPython.display import display

print("CUDA available:", torch.cuda.is_available())


# ============================================================
# CELL 3 — Load GPT-2 Small
# ============================================================

model = tl.HookedTransformer.from_pretrained("gpt2")

print(f"Layers : {model.cfg.n_layers}")
print(f"Heads  : {model.cfg.n_heads}")
print(f"d_model: {model.cfg.d_model}")


# ============================================================
# CELL 4 — Dataset
# ============================================================

dataset = [

    {"prompt": "Alice gave Bob the book because",       "correct": " she", "incorrect": " he"},
    {"prompt": "Bob helped Alice move because",         "correct": " he",  "incorrect": " she"},
    {"prompt": "Sarah thanked John because",            "correct": " he",  "incorrect": " she"},
    {"prompt": "John comforted Sarah because",          "correct": " she", "incorrect": " he"},
    {"prompt": "Emma supported James because",          "correct": " he",  "incorrect": " she"},
    {"prompt": "James encouraged Emma because",         "correct": " she", "incorrect": " he"},

    {"prompt": "When Alice told Bob the news,",         "correct": " he",  "incorrect": " she"},
    {"prompt": "When Bob explained the mistake to Alice,", "correct": " she", "incorrect": " he"},
    {"prompt": "Sarah informed John about the meeting because", "correct": " he", "incorrect": " she"},
    {"prompt": "James reminded Emma about the deadline because", "correct": " she", "incorrect": " he"},

    {"prompt": "Alice hugged Bob because",              "correct": " he",  "incorrect": " she"},
    {"prompt": "Bob admired Alice because",             "correct": " she", "incorrect": " he"},

    {"prompt": "When Alice and Bob arrived at the station, Alice gave the tickets to Bob because",
     "correct": " he", "incorrect": " she"},

    {"prompt": "After Sarah met John at the office, Sarah brought coffee for John because",
     "correct": " he", "incorrect": " she"},

    {"prompt": "When Emma and James finished the project, Emma thanked James because",
     "correct": " he", "incorrect": " she"},

    {"prompt": "After Bob visited Alice at home, Bob cooked dinner for Alice because",
     "correct": " she", "incorrect": " he"},

    {"prompt": "Bob received advice from Alice because",
     "correct": " she", "incorrect": " he"},

    {"prompt": "Alice learned programming from Bob because",
     "correct": " he", "incorrect": " she"},

    {"prompt": "Alice criticized Bob because",
     "correct": " he", "incorrect": " she"},

    {"prompt": "Bob praised Alice because",
     "correct": " she", "incorrect": " he"},
]

print(f"Dataset size: {len(dataset)} prompts")


# ============================================================
# CELL 5 — Logit Lens
# ============================================================

def logit_lens(prompt):

    tokens = model.to_tokens(prompt)

    logits, cache = model.run_with_cache(tokens)

    final_prediction = model.to_string(
        logits[0, -1].argmax()
    )

    print("\n" + "=" * 60)
    print("PROMPT:", prompt)
    print("=" * 60)

    print("\nFinal prediction:", final_prediction)

    print("\nLogit Lens:")

    for layer in range(model.cfg.n_layers):

        resid = cache["resid_post", layer][0, -1]

        logits_at_layer = model.unembed(
            model.ln_final(
                resid.unsqueeze(0).unsqueeze(0)
            )
        )

        top_token = logits_at_layer[0, 0].argmax()

        print(
            f"Layer {layer:2d}: "
            f"{model.to_string(top_token)}"
        )

# Example
logit_lens("Alice gave Bob the book because")


# ============================================================
# CELL 6 — Attention Visualization
# ============================================================

def show_attention(prompt, layer=10):

    tokens = model.to_tokens(prompt)

    _, cache = model.run_with_cache(tokens)

    attn = cache["pattern", layer]

    display(

        cv.attention.attention_patterns(

            tokens=model.to_str_tokens(prompt),

            attention=attn[0]

        )

    )

show_attention(
    "Alice gave Bob the book because",
    layer=10
)


# ============================================================
# CELL 7 — Metrics
# ============================================================

def logit_diff_metric(
    logits,
    correct_token,
    incorrect_token
):

    correct_id = model.to_single_token(
        correct_token
    )

    incorrect_id = model.to_single_token(
        incorrect_token
    )

    return (
        logits[0, -1, correct_id]
        - logits[0, -1, incorrect_id]
    )


def measure_accuracy(dataset, hooks=[]):

    correct = 0

    for item in dataset:

        tokens = model.to_tokens(
            item["prompt"]
        )

        if hooks:

            logits = model.run_with_hooks(

                tokens,

                fwd_hooks=hooks

            )

        else:

            logits, _ = model.run_with_cache(
                tokens
            )

        prediction = model.to_string(
            logits[0, -1].argmax()
        )

        if prediction.strip() == item["correct"].strip():
            correct += 1

    return correct / len(dataset)


def measure_logit_diff(dataset, hooks=[]):

    diffs = []

    for item in dataset:

        tokens = model.to_tokens(
            item["prompt"]
        )

        if hooks:

            logits = model.run_with_hooks(
                tokens,
                fwd_hooks=hooks
            )

        else:

            logits, _ = model.run_with_cache(
                tokens
            )

        diff = logit_diff_metric(

            logits,

            item["correct"],
            item["incorrect"]

        )

        diffs.append(diff.item())

    return sum(diffs) / len(diffs)


baseline_acc = measure_accuracy(dataset)

baseline_diff = measure_logit_diff(dataset)

print("\nBaseline accuracy:", baseline_acc)

print("Baseline logit diff:", baseline_diff)


# ============================================================
# CELL 8 — SAFE Activation Patching
# ============================================================

results_per_prompt = []

valid_examples = 0

for item in dataset:

    clean_prompt = item["prompt"]

    # ----------------------------------------
    # SAME-TOKEN-LENGTH corruption
    # ----------------------------------------

    corrupted_prompt = (
        clean_prompt
        .replace("Alice", "Sarah")
        .replace("Bob", "John")
        .replace("Emma", "Olivia")
        .replace("James", "Michael")
    )

    # ----------------------------------------
    # Tokenize
    # ----------------------------------------

    clean_tokens = model.to_tokens(
        clean_prompt
    )

    corrupted_tokens = model.to_tokens(
        corrupted_prompt
    )

    # ----------------------------------------
    # IMPORTANT:
    # Skip mismatched token lengths
    # ----------------------------------------

    if clean_tokens.shape != corrupted_tokens.shape:

        print("\nSkipping token mismatch:")
        print(clean_prompt)
        print(corrupted_prompt)

        continue

    # ----------------------------------------
    # Run clean prompt
    # ----------------------------------------

    _, clean_cache = model.run_with_cache(
        clean_tokens
    )

    # ----------------------------------------
    # Activation patching
    # ----------------------------------------

    patch_results = patching.get_act_patch_attn_head_out_all_pos(

        model,

        corrupted_tokens,

        clean_cache,

        lambda logits: logit_diff_metric(

            logits,

            item["correct"],

            item["incorrect"]

        )

    )

    results_per_prompt.append(
        patch_results
    )

    valid_examples += 1

print("\nValid examples used:", valid_examples)

avg_patch_results = torch.stack(
    results_per_prompt
).mean(0)

print(
    "\nPatch result shape:",
    avg_patch_results.shape
)


# ============================================================
# CELL 9 — Heatmap
# ============================================================

fig = px.imshow(

    avg_patch_results.detach().cpu().numpy(),

    labels={

        "x": "Head",

        "y": "Layer",

        "color": "Patch Effect"

    },

    title="Average Head Importance Across Dataset",

    color_continuous_scale="RdBu",

    color_continuous_midpoint=0

)

fig.show()


# ============================================================
# CELL 10 — Most Important Heads
# ============================================================

flat = avg_patch_results.flatten()

top_indices = flat.argsort(descending=True)[:5]

top_heads = []

print("\nTop Important Heads:\n")

for idx in top_indices:

    layer = (
        idx // model.cfg.n_heads
    ).item()

    head = (
        idx % model.cfg.n_heads
    ).item()

    score = flat[idx].item()

    top_heads.append((layer, head))

    print(
        f"Layer {layer}, Head {head} "
        f"→ Score {score:.4f}"
    )


# ============================================================
# CELL 11 — Visualize One Important Head
# ============================================================

layer, head = top_heads[0]

print(
    f"\nVisualizing Layer {layer}, Head {head}"
)

prompt = dataset[0]["prompt"]

tokens = model.to_tokens(prompt)

_, cache = model.run_with_cache(tokens)

cv.attention.attention_patterns(

    tokens=model.to_str_tokens(prompt),

    attention=cache[
        "pattern",
        layer
    ][0, head:head+1]

)


# ============================================================
# CELL 12 — Head Ablation
# ============================================================

def ablate_head_hook(layer, head):

    def hook(value, hook):

        value[:, :, head, :] = 0

        return value

    return hook


print("\nSingle Head Ablation:\n")

for layer, head in top_heads:

    hooks = [

        (

            f"blocks.{layer}.attn.hook_v",

            ablate_head_hook(layer, head)

        )

    ]

    acc = measure_accuracy(
        dataset,
        hooks
    )

    diff = measure_logit_diff(
        dataset,
        hooks
    )

    print(

        f"Ablate L{layer}H{head}: "

        f"accuracy={acc:.1%}, "

        f"logit_diff={diff:.4f}"

    )


# ============================================================
# CELL 13 — Full Circuit Ablation
# ============================================================

all_hooks = [

    (

        f"blocks.{layer}.attn.hook_v",

        ablate_head_hook(layer, head)

    )

    for layer, head in top_heads

]

ablated_acc = measure_accuracy(
    dataset,
    all_hooks
)

ablated_diff = measure_logit_diff(
    dataset,
    all_hooks
)

print("\n" + "=" * 60)
print("FULL CIRCUIT ABLATION")
print("=" * 60)

print("\nBaseline Accuracy:")
print(baseline_acc)

print("\nAblated Accuracy:")
print(ablated_acc)

print("\nAccuracy Drop:")
print(
    baseline_acc - ablated_acc
)

print("\nBaseline Logit Diff:")
print(
    baseline_diff
)

print("\nAblated Logit Diff:")
print(
    ablated_diff
)

print("\nPercent Signal Removed:")
print(

    (
        baseline_diff
        - ablated_diff
    )

    / abs(baseline_diff)

)

CUDA available: True


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Layers : 12
Heads  : 12
d_model: 768
Dataset size: 20 prompts

PROMPT: Alice gave Bob the book because

Final prediction:  she

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  they
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she



Baseline accuracy: 0.75
Baseline logit diff: 0.10735387802124023


  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]


Valid examples used: 19

Patch result shape: torch.Size([12, 12])



Top Important Heads:

Layer 11, Head 10 → Score 0.1079
Layer 7, Head 3 → Score 0.1014
Layer 0, Head 3 → Score 0.0999
Layer 10, Head 6 → Score 0.0991
Layer 11, Head 7 → Score 0.0989

Visualizing Layer 11, Head 10

Single Head Ablation:

Ablate L11H10: accuracy=40.0%, logit_diff=0.1244
Ablate L7H3: accuracy=60.0%, logit_diff=0.0465
Ablate L0H3: accuracy=65.0%, logit_diff=0.0997
Ablate L10H6: accuracy=70.0%, logit_diff=0.1124
Ablate L11H7: accuracy=65.0%, logit_diff=0.1256

FULL CIRCUIT ABLATION

Baseline Accuracy:
0.75

Ablated Accuracy:
0.45

Accuracy Drop:
0.3

Baseline Logit Diff:
0.10735387802124023

Ablated Logit Diff:
0.05386896133422851

Percent Signal Removed:
0.4982113145128264


In [ ]:


# ============================================================

# CELL 2 — Imports

# ============================================================

import torch

import transformer_lens as tl

import transformer_lens.patching as patching

import circuitsvis as cv

import plotly.express as px

from IPython.display import display

print("CUDA available:", torch.cuda.is_available())

# ============================================================

# CELL 3 — Load GPT-2 Small

# ============================================================

model = tl.HookedTransformer.from_pretrained("gpt2")

print(f"Layers : {model.cfg.n_layers}")

print(f"Heads  : {model.cfg.n_heads}")

print(f"d_model: {model.cfg.d_model}")

# ============================================================

# CELL 4 — Dataset

# ============================================================

dataset = [

    # Basic transfer / helping

    {"prompt": "Alice gave Bob the book because",       "correct": " she", "incorrect": " he"},

    {"prompt": "Bob helped Alice move because",         "correct": " he",  "incorrect": " she"},

    {"prompt": "Sarah thanked John because",            "correct": " he",  "incorrect": " she"},

    {"prompt": "John comforted Sarah because",          "correct": " she", "incorrect": " he"},

    {"prompt": "Emma supported James because",          "correct": " he",  "incorrect": " she"},

    {"prompt": "James encouraged Emma because",         "correct": " she", "incorrect": " he"},

    # Communication

    {"prompt": "When Alice told Bob the news,",         "correct": " he",  "incorrect": " she"},

    {"prompt": "When Bob explained the mistake to Alice,", "correct": " she", "incorrect": " he"},

    {"prompt": "Sarah informed John about the meeting because", "correct": " he", "incorrect": " she"},

    {"prompt": "James reminded Emma about the deadline because", "correct": " she", "incorrect": " he"},

    # Emotional / social

    {"prompt": "Alice hugged Bob because",              "correct": " he",  "incorrect": " she"},

    {"prompt": "Bob admired Alice because",             "correct": " she", "incorrect": " he"},

    # Long-context

    {"prompt": "When Alice and Bob arrived at the station, Alice gave the tickets to Bob because",

     "correct": " he", "incorrect": " she"},

    {"prompt": "After Sarah met John at the office, Sarah brought coffee for John because",

     "correct": " he", "incorrect": " she"},

    {"prompt": "When Emma and James finished the project, Emma thanked James because",

     "correct": " he", "incorrect": " she"},

    {"prompt": "After Bob visited Alice at home, Bob cooked dinner for Alice because",

     "correct": " she", "incorrect": " he"},

    # Reversed order / harder

    {"prompt": "Bob received advice from Alice because",

     "correct": " she", "incorrect": " he"},

    {"prompt": "Alice learned programming from Bob because",

     "correct": " he", "incorrect": " she"},

    # Ambiguous / reasoning-heavy

    {"prompt": "Alice criticized Bob because",

     "correct": " he", "incorrect": " she"},

    {"prompt": "Bob praised Alice because",

     "correct": " she", "incorrect": " he"},

]

print(f"Dataset size: {len(dataset)} prompts")

# ============================================================

# CELL 5 — Logit Lens

# ============================================================

def logit_lens(prompt):

    tokens = model.to_tokens(prompt)

    logits, cache = model.run_with_cache(tokens)

    final_prediction = model.to_string(

        logits[0, -1].argmax()

    )

    print("\n" + "=" * 60)

    print("PROMPT:", prompt)

    print("=" * 60)

    print("\nFinal prediction:", final_prediction)

    print("\nLogit Lens:")

    for layer in range(model.cfg.n_layers):

        resid = cache["resid_post", layer][0, -1]

        logits_at_layer = model.unembed(

            model.ln_final(

                resid.unsqueeze(0).unsqueeze(0)

            )

        )

        top_token = logits_at_layer[0, 0].argmax()

        print(

            f"Layer {layer:2d}: "

            f"{model.to_string(top_token)}"

        )

logit_lens("Alice gave Bob the book because")

# ============================================================

# CELL 6 — Attention Visualization

# ============================================================

def show_attention(prompt, layer=10):

    tokens = model.to_tokens(prompt)

    _, cache = model.run_with_cache(tokens)

    attn = cache["pattern", layer]

    display(

        cv.attention.attention_patterns(

            tokens=model.to_str_tokens(prompt),

            attention=attn[0]

        )

    )

show_attention(

    "Alice gave Bob the book because",

    layer=10

)

# ============================================================

# CELL 7 — Metric Functions

# ============================================================

def logit_diff_metric(

    logits,

    correct_token,

    incorrect_token

):

    correct_id = model.to_single_token(

        correct_token

    )

    incorrect_id = model.to_single_token(

        incorrect_token

    )

    return (

        logits[0, -1, correct_id]

        - logits[0, -1, incorrect_id]

    )

def measure_accuracy(dataset, hooks=[]):

    correct = 0

    for item in dataset:

        tokens = model.to_tokens(

            item["prompt"]

        )

        if hooks:

            logits = model.run_with_hooks(

                tokens,

                fwd_hooks=hooks

            )

        else:

            logits, _ = model.run_with_cache(

                tokens

            )

        prediction = model.to_string(

            logits[0, -1].argmax()

        )

        if prediction.strip() == item["correct"].strip():

            correct += 1

    return correct / len(dataset)

def measure_logit_diff(dataset, hooks=[]):

    diffs = []

    for item in dataset:

        tokens = model.to_tokens(

            item["prompt"]

        )

        if hooks:

            logits = model.run_with_hooks(

                tokens,

                fwd_hooks=hooks

            )

        else:

            logits, _ = model.run_with_cache(

                tokens

            )

        diff = logit_diff_metric(

            logits,

            item["correct"],

            item["incorrect"]

        )

        diffs.append(diff.item())

    return sum(diffs) / len(diffs)

baseline_acc = measure_accuracy(dataset)

baseline_diff = measure_logit_diff(dataset)

print("\nBaseline accuracy:", baseline_acc)

print("Baseline logit diff:", baseline_diff)

# ============================================================

# CELL 8 — SAFE Activation Patching

# ============================================================

results_per_prompt = []

valid_examples = 0

for item in dataset:

    clean_prompt = item["prompt"]

    # ------------------------------------------------

    # SAFE corruption logic

    # ------------------------------------------------

    corrupted_prompt = (

        item["prompt"]

        .replace("Alice", "TEMP1")

        .replace("Bob", "Alice")

        .replace("TEMP1", "Bob")

        .replace("Sarah", "TEMP2")

        .replace("John", "Sarah")

        .replace("TEMP2", "John")

        .replace("Emma", "TEMP3")

        .replace("James", "Emma")

        .replace("TEMP3", "James")

        .replace("Olivia", "TEMP4")

        .replace("Michael", "Olivia")

        .replace("TEMP4", "Michael")

        .replace("Riley", "TEMP5")

        .replace("Casey", "Riley")

        .replace("TEMP5", "Casey")

    )

    clean_tokens = model.to_tokens(

        clean_prompt

    )

    corrupted_tokens = model.to_tokens(

        corrupted_prompt

    )

    # ------------------------------------------------

    # IMPORTANT

    # ------------------------------------------------

    if clean_tokens.shape != corrupted_tokens.shape:

        print("\nSkipping token mismatch:")

        print(clean_prompt)

        print(corrupted_prompt)

        continue

    _, clean_cache = model.run_with_cache(

        clean_tokens

    )

    patch_results = patching.get_act_patch_attn_head_out_all_pos(

        model,

        corrupted_tokens,

        clean_cache,

        lambda logits: logit_diff_metric(

            logits,

            item["correct"],

            item["incorrect"]

        )

    )

    results_per_prompt.append(

        patch_results

    )

    valid_examples += 1

print("\nValid examples used:", valid_examples)

avg_patch_results = torch.stack(

    results_per_prompt

).mean(0)

print(

    "\nPatch result shape:",

    avg_patch_results.shape

)

# ============================================================

# CELL 9 — Heatmap

# ============================================================

fig = px.imshow(

    avg_patch_results.detach().cpu().numpy(),

    labels={

        "x": "Head",

        "y": "Layer",

        "color": "Patch Effect"

    },

    title="Average Head Importance Across Dataset",

    color_continuous_scale="RdBu",

    color_continuous_midpoint=0

)

fig.show()

# ============================================================

# CELL 10 — Top Heads

# ============================================================

flat = avg_patch_results.flatten()

top_indices = flat.argsort(descending=True)[:5]

top_heads = []

print("\nTop 5 Most Important Heads:\n")

for idx in top_indices:

    layer = (

        idx // model.cfg.n_heads

    ).item()

    head = (

        idx % model.cfg.n_heads

    ).item()

    score = flat[idx].item()

    top_heads.append((layer, head))

    print(

        f"Layer {layer}, Head {head} "

        f"→ Score {score:.4f}"

    )

# ============================================================

# CELL 11 — Visualize Best Head

# ============================================================

layer, head = top_heads[0]

print(

    f"\nVisualizing Layer {layer}, Head {head}"

)

prompt = dataset[0]["prompt"]

tokens = model.to_tokens(prompt)

_, cache = model.run_with_cache(tokens)

cv.attention.attention_patterns(

    tokens=model.to_str_tokens(prompt),

    attention=cache[

        "pattern",

        layer

    ][0, head:head+1]

)

# ============================================================

# CELL 12 — Head Ablation Hook

# ============================================================

def ablate_head_hook(layer, head):

    def hook(value, hook):

        value[:, :, head, :] = 0

        return value

    return hook

# ============================================================

# CELL 13 — Single Head Ablation

# ============================================================

print("\nSingle Head Ablation Results:\n")

for layer, head in top_heads:

    hooks = [

        (

            f"blocks.{layer}.attn.hook_v",

            ablate_head_hook(layer, head)

        )

    ]

    acc = measure_accuracy(

        dataset,

        hooks

    )

    diff = measure_logit_diff(

        dataset,

        hooks

    )

    print(

        f"Ablate L{layer}H{head}: "

        f"accuracy={acc:.1%}, "

        f"logit_diff={diff:.4f}"

    )

# ============================================================

# CELL 14 — Full Circuit Ablation

# ============================================================

all_hooks = [

    (

        f"blocks.{layer}.attn.hook_v",

        ablate_head_hook(layer, head)

    )

    for layer, head in top_heads

]

ablated_acc = measure_accuracy(

    dataset,

    all_hooks

)

ablated_diff = measure_logit_diff(

    dataset,

    all_hooks

)

print("\n" + "=" * 60)

print("FULL CIRCUIT ABLATION")

print("=" * 60)

print("\nBaseline Accuracy:")

print(baseline_acc)

print("\nAblated Accuracy:")

print(ablated_acc)

print("\nAccuracy Drop:")

print(

    baseline_acc - ablated_acc

)

print("\nBaseline Logit Diff:")

print(

    baseline_diff

)

print("\nAblated Logit Diff:")

print(

    ablated_diff

)

print("\nPercent Signal Removed:")

print(

    (

        baseline_diff

        - ablated_diff

    )

    / abs(baseline_diff)

)

# ============================================================

# CELL 15 — Per-Prompt Breakdown

# ============================================================

print("\nPer-Prompt Breakdown:\n")

print(

    f"{'Prompt':<55} "

    f"{'Base':>8} "

    f"{'Ablated':>10} "

    f"{'Drop':>8}"

)

print("-" * 90)

for item in dataset:

    tokens = model.to_tokens(

        item["prompt"]

    )

    logits_base, _ = model.run_with_cache(

        tokens

    )

    diff_base = logit_diff_metric(

        logits_base,

        item["correct"],

        item["incorrect"]

    ).item()

    logits_abl = model.run_with_hooks(

        tokens,

        fwd_hooks=all_hooks

    )

    diff_abl = logit_diff_metric(

        logits_abl,

        item["correct"],

        item["incorrect"]

    ).item()

    drop = diff_base - diff_abl

    short_prompt = (

        item["prompt"][:52] + "..."

        if len(item["prompt"]) > 52

        else item["prompt"]

    )

    print(

        f"{short_prompt:<55} "

        f"{diff_base:>8.3f} "

        f"{diff_abl:>10.3f} "

        f"{drop:>8.3f}"

    )

# ============================================================

# CELL 16 — CAUSAL vs NON-CAUSAL

# ============================================================

causal_prompts = [

    item for item in dataset

    if "because" in item["prompt"]

]

noncausal_prompts = [

    item for item in dataset

    if "because" not in item["prompt"]

]

print("\n" + "=" * 60)

print("CAUSAL vs NON-CAUSAL")

print("=" * 60)

for label, subset in [

    ("Causal ('because')", causal_prompts),

    ("Non-causal", noncausal_prompts)

]:

    if not subset:

        continue

    base = measure_logit_diff(

        subset

    )

    abl = measure_logit_diff(

        subset,

        all_hooks

    )

    pct_removed = (

        (base - abl)

        / abs(base)

        if base != 0

        else 0

    )

    print(f"\n{label}")

    print(f"n = {len(subset)}")

    print(

        f"Baseline logit diff: "

        f"{base:.4f}"

    )

    print(

        f"Ablated logit diff: "

        f"{abl:.4f}"

    )

    print(

        f"Percent signal removed: "

        f"{pct_removed:.1%}"

    )

# ============================================================

# CELL 17 — Fallback Behavior Analysis

# ============================================================

print("\n" + "=" * 60)

print("FALLBACK BEHAVIOR ANALYSIS")

print("=" * 60)

for item in dataset[:8]:

    tokens = model.to_tokens(

        item["prompt"]

    )

    logits_abl = model.run_with_hooks(

        tokens,

        fwd_hooks=all_hooks

    )

    pred = model.to_string(

        logits_abl[0, -1].argmax()

    )

    expected = item["correct"]

    status = (

        "✓"

        if pred.strip() == expected.strip()

        else "✗"

    )

    print(f"\n{status} {item['prompt']}")

    print(

        f"Expected: {expected}"

    )

    print(

        f"Predicted: {pred}"

    )

CUDA available: True


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Layers : 12
Heads  : 12
d_model: 768
Dataset size: 20 prompts

PROMPT: Alice gave Bob the book because

Final prediction:  she

Logit Lens:
Layer  0:  the
Layer  1:  the
Layer  2:  the
Layer  3:  they
Layer  4:  they
Layer  5:  he
Layer  6:  he
Layer  7:  he
Layer  8:  he
Layer  9:  he
Layer 10:  she
Layer 11:  she



Baseline accuracy: 0.75
Baseline logit diff: 0.10735387802124023


  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]


Valid examples used: 17

Patch result shape: torch.Size([12, 12])



Top 5 Most Important Heads:

Layer 8, Head 3 → Score -0.0262
Layer 7, Head 3 → Score -0.0612
Layer 9, Head 5 → Score -0.0734
Layer 4, Head 11 → Score -0.0984
Layer 7, Head 1 → Score -0.0999

Visualizing Layer 8, Head 3

Single Head Ablation Results:

Ablate L8H3: accuracy=65.0%, logit_diff=0.1102
Ablate L7H3: accuracy=60.0%, logit_diff=0.0465
Ablate L9H5: accuracy=70.0%, logit_diff=0.0832
Ablate L4H11: accuracy=75.0%, logit_diff=0.0804
Ablate L7H1: accuracy=75.0%, logit_diff=0.1100

FULL CIRCUIT ABLATION

Baseline Accuracy:
0.75

Ablated Accuracy:
0.55

Accuracy Drop:
0.19999999999999996

Baseline Logit Diff:
0.10735387802124023

Ablated Logit Diff:
-0.03775300979614258

Percent Signal Removed:
1.3516688031397717

Per-Prompt Breakdown:

Prompt                                                      Base    Ablated     Drop
------------------------------------------------------------------------------------------
Alice gave Bob the book because                            0.225      0.408 

In [ ]:
# ============================================================
# Visualize One Important Head
# ============================================================

def show_head_attention(prompt, layer, head):

    tokens = model.to_tokens(prompt)

    _, cache = model.run_with_cache(tokens)

    attn = cache["pattern", layer]

    print(
        f"\nLayer {layer}, Head {head}"
    )

    print(f"Prompt: {prompt}")

    display(

        cv.attention.attention_patterns(

            tokens=model.to_str_tokens(prompt),

            attention=attn[0, head:head+1]

        )

    )


# ============================================================
# Most important head
# ============================================================

layer = 7
head = 3


# ============================================================
# Key prompts
# ============================================================

show_head_attention(

    "Alice gave Bob the book because",

    layer=layer,
    head=head

)

show_head_attention(

    "Bob helped Alice move because",

    layer=layer,
    head=head

)

show_head_attention(

    "When Alice told Bob the news,",

    layer=layer,
    head=head

)


Layer 7, Head 3
Prompt: Alice gave Bob the book because



Layer 7, Head 3
Prompt: Bob helped Alice move because



Layer 7, Head 3
Prompt: When Alice told Bob the news,


In [ ]:
#============================================================
# EXTENSION 1 — Gender-Neutral Dataset
# Fixes the name-gender confound by using neutral names +
# explicit gender markers so the model can't rely on
# name-gender priors.
# ============================================================

neutral_prompts = [
    # (prompt, correct_pronoun, incorrect_pronoun, subject_gender)
    # Format: "[Name], who is a [gender], <verb> [Name2] because"
    # Subject is always the first named person.

    # --- Transfer / giving ---
    ("Alex, who is a woman, gave Jordan the book because", " she", " he"),
    ("Jordan, who is a man, gave Alex the ticket because", " he", " she"),
    ("Riley, who is a woman, handed Casey the keys because", " she", " he"),
    ("Casey, who is a man, passed Riley the note because", " he", " she"),

    # --- Helping ---
    ("Alex, who is a man, helped Jordan move because", " he", " she"),
    ("Jordan, who is a woman, helped Alex carry boxes because", " she", " he"),
    ("Riley, who is a man, assisted Casey with the task because", " he", " she"),
    ("Casey, who is a woman, supported Riley during the crisis because", " she", " he"),

    # --- Communication ---
    ("When Alex, who is a woman, told Jordan the news,", " she", " he"),
    ("When Jordan, who is a man, explained the plan to Alex,", " he", " she"),
    ("After Riley, who is a woman, called Casey about the issue,", " she", " he"),
    ("After Casey, who is a man, wrote Alex the letter,", " he", " she"),

    # --- Emotional / social ---
    ("Alex, who is a man, comforted Jordan because", " he", " she"),
    ("Jordan, who is a woman, encouraged Alex because", " she", " he"),
    ("Riley, who is a man, hugged Casey because", " he", " she"),
    ("Casey, who is a woman, praised Riley because", " she", " he"),

    # --- Long context ---
    ("When Alex, who is a woman, and Jordan arrived at the station, Alex gave the tickets to Jordan because", " she", " he"),
    ("When Jordan, who is a man, and Riley reached the office, Jordan handed the report to Riley because", " he", " she"),

    # --- Reversed order (object-first sentences) ---
    ("Jordan received advice from Alex, who is a woman, because", " she", " he"),
    ("Alex received help from Jordan, who is a man, because", " he", " she"),
]

print(f"Neutral dataset: {len(neutral_prompts)} prompts")
print("\nSample entries:")
for prompt, correct, incorrect in neutral_prompts[:4]:
    print(f"  Prompt:    {prompt}")
    print(f"  Correct:   {correct.strip()} | Incorrect: {incorrect.strip()}\n")


# ============================================================
# Evaluate baseline on neutral dataset
# ============================================================

def evaluate_dataset(prompts):
    """
    Returns accuracy and mean logit diff for a list of
    (prompt, correct_token, incorrect_token) tuples.
    """
    correct_count = 0
    logit_diffs = []

    for prompt, correct_tok, incorrect_tok in prompts:
        tokens = model.to_tokens(prompt)
        logits = model(tokens)[0, -1]  # last token logits

        correct_id   = model.to_single_token(correct_tok)
        incorrect_id = model.to_single_token(incorrect_tok)

        ld = (logits[correct_id] - logits[incorrect_id]).item()
        logit_diffs.append(ld)

        predicted_id = logits.argmax().item()
        if predicted_id == correct_id:
            correct_count += 1

    accuracy = correct_count / len(prompts)
    mean_ld  = sum(logit_diffs) / len(logit_diffs)
    return accuracy, mean_ld, logit_diffs

neutral_acc, neutral_ld, neutral_lds = evaluate_dataset(neutral_prompts)
print(f"\nNeutral dataset baseline:")
print(f"  Accuracy:        {neutral_acc:.1%}")
print(f"  Mean logit diff: {neutral_ld:.4f}")
 # ============================================================
# FIXED COMPARISON CODE
# Converts original dataset into tuple format so it works
# with evaluate_dataset()
# ============================================================

# Convert original dataset into:
# (prompt, correct, incorrect)

original_prompts = [

    (

        item["prompt"],

        item["correct"],

        item["incorrect"]

    )

    for item in dataset

]


# ============================================================
# Evaluate original dataset
# ============================================================

original_acc, original_ld, original_lds = evaluate_dataset(
    original_prompts
)

print(f"\nOriginal dataset baseline:")
print(f"  Accuracy:        {original_acc:.1%}")
print(f"  Mean logit diff: {original_ld:.4f}")


# ============================================================
# Compare neutral vs original
# ============================================================

print("\n" + "=" * 60)
print("DATASET COMPARISON")
print("=" * 60)

print(f"\nOriginal dataset:")
print(f"  Accuracy:        {original_acc:.1%}")
print(f"  Mean logit diff: {original_ld:.4f}")

print(f"\nNeutral dataset:")
print(f"  Accuracy:        {neutral_acc:.1%}")
print(f"  Mean logit diff: {neutral_ld:.4f}")


# ============================================================
# Interpretation
# ============================================================

print("\nInterpretation:")

if neutral_acc >= original_acc - 0.1:

    print(
        "\nNeutral accuracy remained relatively stable."
    )

    print(
        "This suggests the behavior is NOT explained "
        "primarily by simple name-gender priors."
    )

else:

    print(
        "\nNeutral accuracy dropped substantially."
    )

    print(
        "This suggests the original prompts may have "
        "relied heavily on memorized name-gender associations."
    )


if neutral_ld > original_ld:

    print(
        "\nInterestingly, the neutral dataset produced "
        "a larger logit margin."
    )

    print(
        "Explicit gender phrases may help stabilize "
        "entity representations in GPT-2."
    )


print("\nNext recommended step:")
print(
    "Run activation patching + ablation on the "
    "neutral dataset to test whether the SAME "
    "circuit still appears."
)

Neutral dataset: 20 prompts

Sample entries:
  Prompt:    Alex, who is a woman, gave Jordan the book because
  Correct:   she | Incorrect: he

  Prompt:    Jordan, who is a man, gave Alex the ticket because
  Correct:   he | Incorrect: she

  Prompt:    Riley, who is a woman, handed Casey the keys because
  Correct:   she | Incorrect: he

  Prompt:    Casey, who is a man, passed Riley the note because
  Correct:   he | Incorrect: she


Neutral dataset baseline:
  Accuracy:        70.0%
  Mean logit diff: 0.5740

Original dataset baseline:
  Accuracy:        75.0%
  Mean logit diff: 0.1074

DATASET COMPARISON

Original dataset:
  Accuracy:        75.0%
  Mean logit diff: 0.1074

Neutral dataset:
  Accuracy:        70.0%
  Mean logit diff: 0.5740

Interpretation:

Neutral accuracy remained relatively stable.
This suggests the behavior is NOT explained primarily by simple name-gender priors.

Interestingly, the neutral dataset produced a larger logit margin.
Explicit gender phrases may he

In [ ]:
# ============================================================
# FIXED Direct Logit Attribution (DLA)
# Uses hook_z + W_O projection
# ============================================================

import torch

def direct_logit_attribution_head(

    prompt,
    correct_tok,
    incorrect_tok,
    layer=7,
    head=3

):

    tokens = model.to_tokens(prompt)

    final_pos = tokens.shape[1] - 1

    _, cache = model.run_with_cache(tokens)

    # ------------------------------------------------
    # hook_z
    # shape:
    # [batch, seq, head, d_head]
    # ------------------------------------------------

    z = cache["z", layer][0, final_pos, head]

    # ------------------------------------------------
    # Project through W_O
    # W_O shape:
    # [layer, head, d_head, d_model]
    # ------------------------------------------------

    W_O = model.W_O[layer, head]

    head_output = z @ W_O

    # ------------------------------------------------
    # Unembedding
    # ------------------------------------------------

    W_U = model.W_U

    logit_contributions = head_output @ W_U

    correct_id = model.to_single_token(correct_tok)

    incorrect_id = model.to_single_token(incorrect_tok)

    contrib_correct = (
        logit_contributions[correct_id]
        .item()
    )

    contrib_incorrect = (
        logit_contributions[incorrect_id]
        .item()
    )

    contrib_diff = (
        contrib_correct
        - contrib_incorrect
    )

    return (

        contrib_correct,

        contrib_incorrect,

        contrib_diff

    )


# ============================================================
# TEST L7H3
# ============================================================

print("=" * 60)
print("Direct Logit Attribution — L7H3")
print("=" * 60)

test_cases = [

    (

        "Alice gave Bob the book because",

        " she",

        " he"

    ),

    (

        "Bob helped Alice move because",

        " he",

        " she"

    ),

    (

        "When Alice told Bob the news,",

        " she",

        " he"

    ),

    (

        "Alex, who is a woman, gave Jordan the book because",

        " she",

        " he"

    ),

    (

        "Jordan, who is a man, helped Alex move because",

        " he",

        " she"

    ),

]

header = (
    f"{'Prompt':<50} "
    f"{'Correct':>8} "
    f"{'Incorrect':>10} "
    f"{'Diff':>8}"
)

print(header)

print("-" * len(header))

dla_diffs = []

for prompt, correct_tok, incorrect_tok in test_cases:

    c, ic, diff = direct_logit_attribution_head(

        prompt,

        correct_tok,

        incorrect_tok,

        layer=7,
        head=3

    )

    dla_diffs.append(diff)

    print(

        f"{prompt[:48]:<50} "

        f"{c:>8.3f} "

        f"{ic:>10.3f} "

        f"{diff:>8.3f}"

    )

print(
    f"\nMean DLA diff (L7H3): "
    f"{sum(dla_diffs)/len(dla_diffs):.4f}"
)

print("""

Interpretation guide:

  diff > 0
    → L7H3 pushes toward correct pronoun

  diff ≈ 0
    → L7H3 is mostly neutral

  diff < 0
    → L7H3 pushes toward incorrect pronoun

""")


# ============================================================
# DLA Across All Top Heads
# ============================================================

circuit_heads = [

    (7, 1),

    (4, 11),

    (9, 5),

    (7, 3),

    (8, 3)

]

print("\nDLA across all circuit heads:")

print(
    f"{'Head':<12} "
    f"{'Mean DLA diff':>14}"
)

print("-" * 30)

for (l, h) in circuit_heads:

    diffs = []

    for prompt, correct_tok, incorrect_tok in test_cases:

        _, _, diff = direct_logit_attribution_head(

            prompt,

            correct_tok,

            incorrect_tok,

            layer=l,
            head=h

        )

        diffs.append(diff)

    mean_diff = (
        sum(diffs)
        / len(diffs)
    )

    print(

        f"L{l}H{h:<8} "

        f"{mean_diff:>14.4f}"

    )

Direct Logit Attribution — L7H3
Prompt                                              Correct  Incorrect     Diff
-------------------------------------------------------------------------------
Alice gave Bob the book because                      -0.576      0.351   -0.927
Bob helped Alice move because                        -0.101     -1.101    1.000
When Alice told Bob the news,                        -3.282      0.288   -3.569
Alex, who is a woman, gave Jordan the book becau     -0.656     -0.358   -0.297
Jordan, who is a man, helped Alex move because       -1.086     -0.930   -0.156

Mean DLA diff (L7H3): -0.7899


Interpretation guide:

  diff > 0
    → L7H3 pushes toward correct pronoun

  diff ≈ 0
    → L7H3 is mostly neutral

  diff < 0
    → L7H3 pushes toward incorrect pronoun



DLA across all circuit heads:
Head          Mean DLA diff
------------------------------
L7H1                0.0855
L4H11               0.6556
L9H5               -0.5388
L7H3               -0.7899
L8H3

In [ ]:
# ============================================================
# EXTENSION 3 — Full Circuit Ablation
# Original vs Neutral Dataset (FULL FIXED VERSION)
# ============================================================

import torch


# ============================================================
# Convert original dataset into tuple format
# ============================================================

original_prompts = [

    (

        item["prompt"],

        item["correct"],

        item["incorrect"]

    )

    for item in dataset

]


# ============================================================
# Hook creator
# ============================================================

def make_ablation_hook(head_idx):

    def hook_fn(value, hook):

        # value shape:
        # [batch, seq, n_heads, d_head]

        value[:, :, head_idx, :] = 0.0

        return value

    return hook_fn


# ============================================================
# Evaluation helper
# ============================================================

def ablate_heads_and_evaluate(

    prompts,

    heads_to_ablate

):

    correct_count = 0

    logit_diffs = []

    hooks = [

        (

            f"blocks.{layer}.attn.hook_z",

            make_ablation_hook(head)

        )

        for layer, head in heads_to_ablate

    ]

    for prompt, correct_tok, incorrect_tok in prompts:

        tokens = model.to_tokens(prompt)

        with model.hooks(fwd_hooks=hooks):

            logits = model(tokens)[0, -1]

        correct_id = model.to_single_token(
            correct_tok
        )

        incorrect_id = model.to_single_token(
            incorrect_tok
        )

        ld = (

            logits[correct_id]

            - logits[incorrect_id]

        ).item()

        logit_diffs.append(ld)

        if logits.argmax().item() == correct_id:

            correct_count += 1

    accuracy = (
        correct_count / len(prompts)
    )

    mean_ld = (
        sum(logit_diffs)
        / len(logit_diffs)
    )

    return accuracy, mean_ld


# ============================================================
# Circuit heads from earlier patching
# ============================================================

circuit_heads = [

    (7, 1),

    (4, 11),

    (9, 5),

    (7, 3),

    (8, 3)

]


# ============================================================
# Main comparison
# ============================================================

print("=" * 60)
print("Circuit Ablation — Original vs Neutral")
print("=" * 60)


# ------------------------------------------------
# Baseline performance
# ------------------------------------------------

orig_base_acc, orig_base_ld, _ = evaluate_dataset(
    original_prompts
)

neutral_base_acc, neutral_base_ld, _ = evaluate_dataset(
    neutral_prompts
)


# ------------------------------------------------
# Ablated performance
# ------------------------------------------------

orig_abl_acc, orig_abl_ld = (

    ablate_heads_and_evaluate(

        original_prompts,

        circuit_heads

    )

)

neutral_abl_acc, neutral_abl_ld = (

    ablate_heads_and_evaluate(

        neutral_prompts,

        circuit_heads

    )

)


# ============================================================
# Print comparison table
# ============================================================

print(

    f"\n{'Dataset':<24}"

    f"{'Base Acc':>12}"

    f"{'Abl Acc':>12}"

    f"{'Δ Acc':>10}"

    f"{'Base LD':>12}"

    f"{'Abl LD':>12}"

    f"{'Δ LD':>12}"

)

print("-" * 94)

orig_delta_acc = (
    orig_abl_acc - orig_base_acc
)

orig_delta_ld = (
    orig_abl_ld - orig_base_ld
)

neutral_delta_acc = (
    neutral_abl_acc - neutral_base_acc
)

neutral_delta_ld = (
    neutral_abl_ld - neutral_base_ld
)

print(

    f"{'Original':<24}"

    f"{orig_base_acc:>12.1%}"

    f"{orig_abl_acc:>12.1%}"

    f"{orig_delta_acc:>+10.1%}"

    f"{orig_base_ld:>12.4f}"

    f"{orig_abl_ld:>12.4f}"

    f"{orig_delta_ld:>+12.4f}"

)

print(

    f"{'Neutral (no confound)':<24}"

    f"{neutral_base_acc:>12.1%}"

    f"{neutral_abl_acc:>12.1%}"

    f"{neutral_delta_acc:>+10.1%}"

    f"{neutral_base_ld:>12.4f}"

    f"{neutral_abl_ld:>12.4f}"

    f"{neutral_delta_ld:>+12.4f}"

)


# ============================================================
# Interpretation
# ============================================================

print("""

Interpretation:

Original dataset:
    Ablation strongly damages performance
    and inverts pronoun preference.

Neutral dataset:
    Ablation does NOT damage performance.
    In fact, logit margin slightly improves.

This suggests the originally identified
circuit is tied more strongly to implicit
name/gender heuristics than to fully
general grammatical subject tracking.

GPT-2 may switch mechanisms when explicit
gender information ("who is a woman/man")
is available.

""")


# ============================================================
# Fallback behavior analysis
# ============================================================

print("\n" + "=" * 60)
print("Fallback Behavior — Neutral Dataset")
print("=" * 60)

print(

    f"\n{'Prompt':<55}"

    f"{'Expected':>10}"

    f"{'Predicted':>12}"

)

print("-" * 80)

hooks = [

    (

        f"blocks.{layer}.attn.hook_z",

        make_ablation_hook(head)

    )

    for layer, head in circuit_heads

]

for prompt, correct_tok, incorrect_tok in neutral_prompts[:10]:

    tokens = model.to_tokens(prompt)

    with model.hooks(fwd_hooks=hooks):

        logits = model(tokens)[0, -1]

    predicted = model.tokenizer.decode(

        [logits.argmax().item()]

    )

    match = (

        "✓"

        if predicted.strip() == correct_tok.strip()

        else "✗"

    )

    short_prompt = (

        prompt[:52] + "..."

        if len(prompt) > 52

        else prompt

    )

    print(

        f"{match} "

        f"{short_prompt:<54}"

        f"{correct_tok.strip():>10}"

        f"{predicted.strip():>12}"

    )


# ============================================================
# Final takeaway
# ============================================================

print("\n" + "=" * 60)
print("Key Scientific Takeaway")
print("=" * 60)

print("""

The original circuit discovery was real:
ablating the heads strongly disrupted
performance on implicit-gender prompts.

However, the neutral control experiment
shows the same circuit does not generalize
cleanly when explicit gender information
is introduced.

This suggests:
    GPT-2 may use different mechanisms for:

    1. implicit gender heuristics
       (Alice → she)

    vs.

    2. explicit semantic gender parsing
       ("who is a woman")

This substantially revises the original
mechanistic interpretation and highlights
the importance of confound-controlled
datasets in interpretability research.

""")

Circuit Ablation — Original vs Neutral

Dataset                     Base Acc     Abl Acc     Δ Acc     Base LD      Abl LD        Δ LD
----------------------------------------------------------------------------------------------
Original                       75.0%       55.0%    -20.0%      0.1074     -0.0378     -0.1451
Neutral (no confound)          70.0%       75.0%     +5.0%      0.5740      0.7419     +0.1679


Interpretation:

Original dataset:
    Ablation strongly damages performance
    and inverts pronoun preference.

Neutral dataset:
    Ablation does NOT damage performance.
    In fact, logit margin slightly improves.

This suggests the originally identified
circuit is tied more strongly to implicit
name/gender heuristics than to fully
general grammatical subject tracking.

GPT-2 may switch mechanisms when explicit
gender information ("who is a woman/man")
is available.



Fallback Behavior — Neutral Dataset

Prompt                                                   Expect

In [23]:
# ============================================================
# EXTENSION 4 — Activation Patching on Neutral Dataset
# ============================================================

import torch
import plotly.express as px

from transformer_lens.patching import (
    get_act_patch_attn_head_out_all_pos
)


# ============================================================
# IMPORTANT:
# Replace woman/man with female/male
# for safer token-length matching
# ============================================================

neutral_prompts_fixed = [

    (

        p.replace(

            "who is a woman",

            "who is female"

        ).replace(

            "who is a man",

            "who is male"

        ),

        c,

        ic

    )

    for (p, c, ic) in neutral_prompts

]


# ============================================================
# Corruption function
# ============================================================

def make_corrupted_neutral(

    prompt,

    correct_tok,

    incorrect_tok

):

    corrupted = prompt

    # ------------------------------------------------
    # Swap explicit gender markers
    # ------------------------------------------------

    corrupted = corrupted.replace(

        "who is female",

        "TEMP_GENDER"

    )

    corrupted = corrupted.replace(

        "who is male",

        "who is female"

    )

    corrupted = corrupted.replace(

        "TEMP_GENDER",

        "who is male"

    )

    # ------------------------------------------------
    # Swap names
    # ------------------------------------------------

    name_pairs = [

        ("Alex", "Jordan"),

        ("Riley", "Casey")

    ]

    for n1, n2 in name_pairs:

        if n1 in corrupted and n2 in corrupted:

            corrupted = corrupted.replace(

                n1,

                "TEMP_NAME"

            )

            corrupted = corrupted.replace(
                n2,
                n1
            )

            corrupted = corrupted.replace(
                "TEMP_NAME",
                n2
            )

    return (

        corrupted,

        incorrect_tok,

        correct_tok

    )


# ============================================================
# Test corruption
# ============================================================

sample = neutral_prompts_fixed[0]

orig_p, c, ic = sample

corr_p, corr_c, corr_ic = make_corrupted_neutral(

    orig_p,

    c,

    ic

)

print("=" * 60)
print("Corruption test")
print("=" * 60)

print("Original:")
print(orig_p)
print("→", c)

print("\nCorrupted:")
print(corr_p)
print("→", corr_c)


# ============================================================
# Metric function
# IMPORTANT: return tensor, not float
# ============================================================

def logit_diff_metric(

    logits,

    correct_id,

    incorrect_id

):

    return (

        logits[0, -1, correct_id]

        - logits[0, -1, incorrect_id]

    )


# ============================================================
# Create GPU tensor
# ============================================================

neutral_patch_scores = torch.zeros(

    model.cfg.n_layers,

    model.cfg.n_heads,

    device=next(model.parameters()).device

)


# ============================================================
# Run patching
# ============================================================

print("\n" + "=" * 60)
print("Running activation patching on neutral dataset...")
print("=" * 60)

valid_count = 0

for prompt, correct_tok, incorrect_tok in neutral_prompts_fixed:

    corrupted_prompt, corr_correct, corr_incorrect = (

        make_corrupted_neutral(

            prompt,

            correct_tok,

            incorrect_tok

        )

    )

    clean_tokens = model.to_tokens(prompt)

    corrupted_tokens = model.to_tokens(
        corrupted_prompt
    )

    # ------------------------------------------------
    # Ensure token lengths match
    # ------------------------------------------------

    if clean_tokens.shape != corrupted_tokens.shape:

        print("\nSkipping token mismatch:")
        print(prompt)
        print(corrupted_prompt)

        continue

    correct_id = model.to_single_token(
        correct_tok
    )

    incorrect_id = model.to_single_token(
        incorrect_tok
    )

    _, clean_cache = model.run_with_cache(
        clean_tokens
    )

    patch_results = (

        get_act_patch_attn_head_out_all_pos(

            model,

            corrupted_tokens,

            clean_cache,

            lambda logits:

                logit_diff_metric(

                    logits,

                    correct_id,

                    incorrect_id

                )

        )

    )

    neutral_patch_scores += patch_results

    valid_count += 1


# ============================================================
# Average results
# ============================================================

neutral_patch_scores /= valid_count

print(f"\nValid prompts used: {valid_count}")
# ============================================================
# Plot heatmap
# ============================================================

import plotly.express as px

fig = px.imshow(
    neutral_patch_scores.cpu().numpy(),
    labels=dict(
        x="Head",
        y="Layer",
        color="Patch effect"
    ),
    title=(
        "Activation patching on neutral dataset "
        "(explicit gender condition)"
    ),
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0
)

# Save as PNG first (works even if fig.show() doesn't render)
try:
    import kaleido
except ImportError:
    !pip install kaleido -q

fig.write_image("neutral_patching_heatmap.png")
print("Saved: neutral_patching_heatmap.png")

fig.show()


# ============================================================
# Top heads
# ============================================================

flat = neutral_patch_scores.flatten()

topk = flat.topk(

    5,

    largest=False

)

print("\n" + "=" * 60)
print("Top 5 causal heads — neutral dataset")
print("=" * 60)

print(

    f"{'Rank':<6}"

    f"{'Layer':<8}"

    f"{'Head':<8}"

    f"{'Score':>10}"

)

print("-" * 36)

neutral_top_heads = []

for rank, (score, idx) in enumerate(

    zip(topk.values, topk.indices),

    1

):

    l, h = divmod(

        idx.item(),

        model.cfg.n_heads

    )

    neutral_top_heads.append((l, h))

    print(

        f"{rank:<6}"

        f"{l:<8}"

        f"{h:<8}"

        f"{score.item():>10.4f}"

    )


# ============================================================
# Compare with original heads
# ============================================================

print("\n" + "=" * 60)
print("Comparison with original circuit")
print("=" * 60)

print("\nOriginal circuit heads:")

original_heads = [

    (7, 1),

    (4, 11),

    (9, 5),

    (7, 3),

    (8, 3)

]

for h in original_heads:
    print(h)

print("\nNeutral dataset heads:")

for h in neutral_top_heads:
    print(h)


# ============================================================
# Mechanistic interpretation
# ============================================================

print("\n" + "=" * 60)
print("Interpretation")
print("=" * 60)

print("""

If the neutral dataset identifies a
different set of heads than the original
dataset, this suggests GPT-2 switches
mechanisms between:

1. implicit gender heuristics
   (Alice → she)

and

2. explicit semantic gender parsing
   ("who is female")

This would explain why:
- original circuit ablation strongly
  disrupted behavior

while:
- neutral dataset ablation had little
  or no negative effect.

The key next step would be:
- ablating the NEW neutral heads
- testing whether they specifically
  control explicit-gender pronoun
  resolution.

""")

Corruption test
Original:
Alex, who is female, gave Jordan the book because
→  she

Corrupted:
Jordan, who is male, gave Alex the book because
→  he

Running activation patching on neutral dataset...


  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

...

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]

  0%|          | 0/144 [00:00<?, ?it/s]


Valid prompts used: 20
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 6.3 MB/s eta 0:00:00


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [25]:
import plotly.io as pio

pio.write_html(fig, "neutral_patching_heatmap.html")
print("Saved as HTML")

Saved as HTML


In [ ]:
# ============================================================
# EXTENSIONS 7, 8, 9
# ============================================================

import torch


# ============================================================
# IMPORTANT:
# Use tuple-format prompts everywhere
# ============================================================

original_prompts = [

    (

        item["prompt"],

        item["correct"],

        item["incorrect"]

    )

    for item in dataset

]

neutral_prompts_fixed = [

    (

        p.replace(

            "who is a woman",

            "who is female"

        ).replace(

            "who is a man",

            "who is male"

        ),

        c,

        ic

    )

    for (p, c, ic) in neutral_prompts

]


# ============================================================
# Shared / specialized head groups
# ============================================================

shared_heads = [

    (9, 5),

    (7, 3)

]

implicit_only = [

    (7, 1),

    (4, 11)

]

explicit_only = [

    (7, 6),

    (5, 11),

    (8, 9)

]

full_original = [

    (7, 1),

    (4, 11),

    (9, 5),

    (7, 3),

    (8, 3)

]

full_neutral = [

    (9, 5),

    (7, 3),

    (7, 6),

    (5, 11),

    (8, 9)

]


# ============================================================
# Conditions
# ============================================================

conditions = [

    (

        "Shared core only",

        shared_heads

    ),

    (

        "Implicit-only heads",

        implicit_only

    ),

    (

        "Explicit-only heads",

        explicit_only

    ),

    (

        "Full original circuit",

        full_original

    ),

    (

        "Full neutral circuit",

        full_neutral

    ),

]


# ============================================================
# Baselines
# ============================================================

orig_base_acc, orig_base_ld, _ = evaluate_dataset(
    original_prompts
)

neut_base_acc, neut_base_ld, _ = evaluate_dataset(
    neutral_prompts_fixed
)


# ============================================================
# EXTENSION 7
# Targeted ablation analysis
# ============================================================

print("=" * 90)
print("Targeted Ablation — Mechanism Dissociation")
print("=" * 90)

print(

    f"\n{'Heads ablated':<28}"

    f"{'Orig LD':>14}"

    f"{'Δ Orig':>12}"

    f"{'Neut LD':>14}"

    f"{'Δ Neut':>12}"

)

print("-" * 84)

for label, heads in conditions:

    orig_acc, orig_ld = (

        ablate_heads_and_evaluate(

            original_prompts,

            heads

        )

    )

    neut_acc, neut_ld = (

        ablate_heads_and_evaluate(

            neutral_prompts_fixed,

            heads

        )

    )

    d_orig = (
        orig_ld - orig_base_ld
    )

    d_neut = (
        neut_ld - neut_base_ld
    )

    print(

        f"{label:<28}"

        f"{orig_ld:>14.4f}"

        f"{d_orig:>+12.4f}"

        f"{neut_ld:>14.4f}"

        f"{d_neut:>+12.4f}"

    )

print("""

Interpretation:

Shared-core ablation:
    Should damage BOTH datasets
    if these heads implement generalized
    entity/pronoun tracking.

Implicit-only ablation:
    Should mainly damage original prompts
    if these heads specialize for
    name-gender heuristics.

Explicit-only ablation:
    Should mainly damage neutral prompts
    if these heads specialize for
    explicit semantic gender parsing.

""")


# ============================================================
# EXTENSION 8
# DLA on new neutral heads
# ============================================================

print("=" * 60)
print("DLA — Neutral-circuit heads")
print("=" * 60)

new_heads = [

    (7, 6),

    (5, 11),

    (8, 9)

]

for layer, head in new_heads:

    diffs = []

    for prompt, correct_tok, incorrect_tok in neutral_prompts_fixed:

        _, _, diff = direct_logit_attribution_head(

            prompt,

            correct_tok,

            incorrect_tok,

            layer=layer,

            head=head

        )

        diffs.append(diff)

    mean_diff = (
        sum(diffs)
        / len(diffs)
    )

    if mean_diff > 0.1:

        direction = "→ pushes CORRECT"

    elif mean_diff < -0.1:

        direction = "→ pushes WRONG"

    else:

        direction = "→ approximately neutral"

    print(

        f"L{layer}H{head:<4} "

        f"mean DLA: {mean_diff:>8.4f}  "

        f"{direction}"

    )


# ============================================================
# EXTENSION 9
# Attention visualization
# ============================================================

print("\n" + "=" * 60)
print("Attention patterns — Shared-core heads")
print("=" * 60)

neutral_viz = [

    "Alex, who is female, gave Jordan the book because",

    "Jordan, who is male, helped Alex move because",

]

for layer, head in shared_heads:

    print("\n" + "-" * 40)

    print(f"L{layer}H{head}")

    print("-" * 40)

    for p in neutral_viz:

        print("\nPrompt:")
        print(p)

        show_head_attention(

            p,

            layer=layer,

            head=head

        )


# ============================================================
# Final mechanistic interpretation
# ============================================================

print("\n" + "=" * 60)
print("Current Mechanistic Hypothesis")
print("=" * 60)

print("""

Current evidence suggests GPT-2 may use a
partially shared pronoun-resolution system.

Shared heads:
    L9H5 and L7H3

These appear across BOTH:
    - implicit gender prompts
    - explicit gender prompts

This suggests they may implement
generalized entity/pronoun tracking.

Condition-specific heads:

Implicit-gender heads:
    L7H1, L4H11, L8H3

Explicit-gender heads:
    L7H6, L5H11, L8H9

These may implement:
    - name/gender heuristics
    - semantic gender parsing

respectively.

This points toward:
    a hybrid circuit architecture
rather than:
    a single monolithic pronoun circuit.

""")

Targeted Ablation — Mechanism Dissociation

Heads ablated                      Orig LD      Δ Orig       Neut LD      Δ Neut
------------------------------------------------------------------------------------
Shared core only                    0.0226     -0.0848        0.3428     +0.0517
Implicit-only heads                 0.0821     -0.0252        0.2646     -0.0265
Explicit-only heads                 0.1431     +0.0358        0.3231     +0.0320
Full original circuit              -0.0378     -0.1451        0.3585     +0.0674
Full neutral circuit                0.0565     -0.0508        0.4083     +0.1172


Interpretation:

Shared-core ablation:
    Should damage BOTH datasets
    if these heads implement generalized
    entity/pronoun tracking.

Implicit-only ablation:
    Should mainly damage original prompts
    if these heads specialize for
    name-gender heuristics.

Explicit-only ablation:
    Should mainly damage neutral prompts
    if these heads specialize for
    explicit 


Prompt:
Jordan, who is male, helped Alex move because

Layer 9, Head 5
Prompt: Jordan, who is male, helped Alex move because



----------------------------------------
L7H3
----------------------------------------

Prompt:
Alex, who is female, gave Jordan the book because

Layer 7, Head 3
Prompt: Alex, who is female, gave Jordan the book because



Prompt:
Jordan, who is male, helped Alex move because

Layer 7, Head 3
Prompt: Jordan, who is male, helped Alex move because



Current Mechanistic Hypothesis


Current evidence suggests GPT-2 may use a
partially shared pronoun-resolution system.

Shared heads:
    L9H5 and L7H3

These appear across BOTH:
    - implicit gender prompts
    - explicit gender prompts

This suggests they may implement
generalized entity/pronoun tracking.

Condition-specific heads:

Implicit-gender heads:
    L7H1, L4H11, L8H3

Explicit-gender heads:
    L7H6, L5H11, L8H9

These may implement:
    - name/gender heuristics
    - semantic gender parsing

respectively.

This points toward:
    a hybrid circuit architecture
rather than:
    a single monolithic pronoun circuit.


